# Rerank em busca híbrida: RRF e WLF (no mesmo corpus)

Este notebook continua a ideia do anterior (**consulta léxica vs semântica**) e adiciona uma etapa essencial em sistemas de busca modernos:

- **Gerar candidatos** com dois “motores” (léxico e semântico)
- **Fazer rerank / fusão** desses resultados para produzir um ranking final

Vamos implementar, do zero e de forma didática, duas técnicas muito usadas:

- **RRF — Reciprocal Rank Fusion** (fusão baseada em posições no ranking)
- **WLF — Weighted Linear Fusion** (fusão linear ponderada de scores normalizados)

> Nota prática: na sua aplicação real, o **Elasticsearch** entrega as listas léxica e semântica.  
> Aqui, vamos **simular** essas listas localmente (BM25 e embeddings) para focar no *rerank*.



## Como executar este notebook (Jupyter / Colab)

Um notebook Jupyter é um arquivo `.ipynb` que mistura:
- **células de texto (Markdown)** para explicações
- **células de código** para executar Python passo a passo

No **Google Colab**:
1. Abra o arquivo `.ipynb` no Colab
2. No menu **Runtime / Ambiente de execução**, escolha **Run all / Executar tudo**
3. Se uma célula começar com `!pip install ...`, execute-a para instalar as dependências
4. Você pode editar qualquer célula e reexecutar (Shift+Enter)

Dica: se algo der erro por falta de pacote, rode novamente a célula de instalação.



## Visão geral do que vamos construir

### Pipeline (simplificado)

1) **Corpus**: uma lista de “documentos” (aqui, frases curtas; na prática, *chunks*)  
2) **Busca léxica (BM25)** → ranking A  
3) **Busca semântica (embeddings + cosseno)** → ranking B  
4) **Fusão / rerank**: combina A e B → ranking final

### Por que fazer rerank?

- A busca **léxica** é ótima para *termos exatos*, siglas, códigos e nomes próprios
- A busca **semântica** é ótima para *paráfrases*, sinônimos e “mesma ideia com palavras diferentes”
- Em sistemas reais, as duas concordam em muitos casos — mas quando discordam, o **rerank** define quem “vence”



## Teoria curta: RRF (Reciprocal Rank Fusion)

A RRF combina rankings usando apenas **a posição (rank)**, sem depender da escala dos scores.

Para cada documento `d`, o score final é:

$$
\text{RRF}(d)=\sum_i \frac{1}{k + \text{rank}_i(d)}
$$

- `rank_i(d)` é a posição do documento no ranking `i` (1 = melhor)
- `k` é um hiperparâmetro (ex.: 60) que suaviza a contribuição

**Vantagens**
- Não precisa calibrar scores (ótimo quando léxico e semântico têm escalas diferentes)
- Bem robusto na prática

**Intuição**: estar no topo de *qualquer* ranking já ajuda bastante.



## Teoria curta: WLF (Weighted Linear Fusion)

A WLF combina rankings usando os **scores**, mas precisa de um passo importante:

1) **Normalizar** os scores de cada ranking (ex.: dividir pelo máximo da lista)  
2) Combinar com um peso `alpha`:

$$
\mathrm{WLF}(d)=\alpha\,\mathrm{norm}_{\mathrm{sem}}(d) + (1-\alpha)\,\mathrm{norm}_{\mathrm{lex}}(d)
$$

- `alpha` controla o quanto você “confia” na busca semântica
  - `alpha = 0.7` → semântica pesa mais
  - `alpha = 0.3` → léxica pesa mais

**Vantagens**
- Dá controle fino via `alpha`
- Pode funcionar muito bem quando os scores são “comparáveis” após normalização

**Atenção**
- Se a normalização for ruim, a fusão pode ficar instável



In [7]:
# Instala dependências (no Colab isso pode levar ~1-2 minutos na primeira vez)
# !pip -q install sentence-transformers

import numpy as np
import pandas as pd

# from IPython.display import display
from sentence_transformers import SentenceTransformer, util


## 1) Corpus (nossos “documentos”)

Na sua aplicação real, cada documento vira vários **chunks**.

Aqui, vamos tratar **cada frase como se fosse um chunk** (um item recuperável).  
Além dos exemplos do notebook anterior, adicionamos alguns itens com **códigos/termos exatos** para ver a diferença no rerank.



In [8]:
# Nosso corpus: cada item é um "chunk" indexável
sentences = [
    # Gato/sofá (paráfrases + inglês)
    "O gato está dormindo no sofá",
    "Um gato dorme em cima do sofá",
    "Há um gato dormindo no sofá da sala",
    "The cat is sleeping on the sofa",

    # Cachorro/parque (paráfrases + inglês)
    "O cachorro correu pelo parque",
    "Um cão estava correndo no parque",
    "O animal correu em um parque público",
    "The dog is running in the park",

    # Matemática
    "Eu gosto de estudar matemática",
    "Aprender matemática é importante",
    "Matemática é uma disciplina difícil",
    "Ensinar matemática exige paciência",

    # Banco (ambiguidade)
    "O banco fechou mais cedo hoje",                    # instituição
    "Sentei no banco da praça para descansar",          # assento
    "O banco central aumentou a taxa de juros",         # instituição

    # Termos exatos / códigos (bons para busca léxica)
    "Manual do produto XJ-900: como calibrar e atualizar o firmware",
    "Erro E42 no sensor: verifique a conexão e reinicie o dispositivo",
    "Para subir o serviço: rode docker compose up -d no diretório do projeto",
    "O contrato referência é o CT-2026-041, assinado em janeiro",

    # Outros tópicos aleatórios
    "A previsão do tempo indica chuva amanhã",
    "Comprei um novo teclado mecânico",
    "A fotossíntese ocorre nas folhas das plantas",
    "The stock market closed higher today",
]

docs = pd.DataFrame({
    "doc_id": range(len(sentences)),
    "chunk_id": [f"{i}_0" for i in range(len(sentences))],  # simulando chunk único por doc
    "title": [f"doc_{i}" for i in range(len(sentences))],
    "content": sentences,
})

docs.head(10)


,doc_id,chunk_id,title,content
0,0,0_0,doc_0,O gato está dormindo no sofá
1,1,1_0,doc_1,Um gato dorme em cima do sofá
2,2,2_0,doc_2,Há um gato dormindo no sofá da sala
3,3,3_0,doc_3,The cat is sleeping on the sofa
4,4,4_0,doc_4,O cachorro correu pelo parque
5,5,5_0,doc_5,Um cão estava correndo no parque
6,6,6_0,doc_6,O animal correu em um parque público
7,7,7_0,doc_7,The dog is running in the park
8,8,8_0,doc_8,Eu gosto de estudar matemática
9,9,9_0,doc_9,Aprender matemática é importante


## 2) Busca léxica (BM25)

A ideia: transformar texto em **tokens** e usar uma função de relevância baseada em:
- frequência do termo no documento (TF)
- raridade do termo no corpus (IDF)
- normalização por tamanho do documento

Não vamos usar Elasticsearch aqui. Vamos implementar uma BM25 “enxuta” só para gerar um ranking léxico.



In [9]:
import re
import unicodedata
from collections import defaultdict, Counter

def normalize_text(text: str) -> str:
    """Normaliza texto: minúsculas + remove acentos."""
    text = text.lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    return text

def tokenize(text: str):
    """Tokeniza: retorna lista de tokens alfanuméricos."""
    text = normalize_text(text)
    return re.findall(r"[a-z0-9]+", text)

# Exemplo rápido
for s in docs["content"].iloc[:4]:
    print(s, "->", tokenize(s))


O gato está dormindo no sofá -> ['o', 'gato', 'esta', 'dormindo', 'no', 'sofa']
Um gato dorme em cima do sofá -> ['um', 'gato', 'dorme', 'em', 'cima', 'do', 'sofa']
Há um gato dormindo no sofá da sala -> ['ha', 'um', 'gato', 'dormindo', 'no', 'sofa', 'da', 'sala']
The cat is sleeping on the sofa -> ['the', 'cat', 'is', 'sleeping', 'on', 'the', 'sofa']


In [10]:
import math

class BM25Retriever:
    """Implementação de BM25 (Okapi)."""

    def __init__(self, k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b = b

        self.texts = None
        self.tokenized_docs = None
        self.doc_term_freqs = None
        self.inverted_index = None
        self.doc_lens = None
        self.avgdl = None
        self.df = None
        self.idf = None
        self.N = 0

    def fit(self, texts):
        """Indexa os documentos."""
        self.texts = list(texts)
        self.tokenized_docs = [tokenize(t) for t in self.texts]
        self.N = len(self.texts)

        self.doc_term_freqs = []
        self.inverted_index = defaultdict(list)
        self.doc_lens = []
        self.df = Counter()

        for doc_id, terms in enumerate(self.tokenized_docs):
            tf = Counter(terms)
            self.doc_term_freqs.append(tf)
            self.doc_lens.append(len(terms))

            for term in tf.keys():
                self.df[term] += 1
                self.inverted_index[term].append(doc_id)

        self.avgdl = sum(self.doc_lens) / max(1, self.N)

        # IDF padrão BM25
        self.idf = {}
        for term, df in self.df.items():
            self.idf[term] = math.log(1 + (self.N - df + 0.5) / (df + 0.5))

        return self

    def _score_doc(self, doc_id: int, query_terms):
        score = 0.0
        tf = self.doc_term_freqs[doc_id]
        dl = self.doc_lens[doc_id] or 1

        for term in query_terms:
            if term not in tf:
                continue

            freq = tf[term]
            idf = self.idf.get(term, 0.0)

            denom = freq + self.k1 * (1 - self.b + self.b * (dl / self.avgdl))
            score += idf * (freq * (self.k1 + 1)) / denom

        return score

    def retrieve(self, query: str, top_k: int = 10):
        """Retorna lista de (score, doc_index)."""
        query_terms = tokenize(query)
        if not query_terms:
            return []

        # Opcional: pega só docs que têm pelo menos um termo (acelera em corpus grande)
        candidates = set()
        for term in set(query_terms):
            candidates.update(self.inverted_index.get(term, []))

        scored = []
        for doc_id in candidates:
            s = self._score_doc(doc_id, query_terms)
            if s > 0:
                scored.append((float(s), doc_id))

        scored.sort(reverse=True, key=lambda x: x[0])
        return scored[:top_k]

# Treina o BM25 no corpus
bm25 = BM25Retriever(k1=1.5, b=0.75).fit(docs["content"].tolist())


## 3) Busca semântica (embeddings + similaridade de cosseno)

Aqui usamos um modelo de embeddings local (Sentence-Transformers) para:
- transformar textos em vetores
- comparar consulta e documentos via **cosseno**
- ranquear os mais similares

Na sua aplicação real, você usa embeddings da OpenAI.  
Para fins didáticos (e para rodar sem chave), usaremos um modelo aberto.



In [11]:
# Carrega o modelo de embeddings
# Observação: o download do modelo pode demorar na primeira execução.
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2")

# Calcula embeddings dos documentos
doc_embeddings = model.encode(
    docs["content"].tolist(),
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

print("Nº de chunks:", len(docs))
print("Dimensão do embedding:", doc_embeddings.shape[1])


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6360.33it/s]
XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Nº de chunks: 23
Dimensão do embedding: 768


In [12]:
def torch_topk_indices(scores_tensor, k: int):
    """Retorna índices dos k maiores valores de um tensor 1D."""
    import torch
    k = min(k, int(scores_tensor.shape[0]))
    values, indices = torch.topk(scores_tensor, k=k)
    return indices.cpu().numpy().tolist()

def semantic_retrieve(query: str, top_k: int = 10):
    """Retorna lista de (score, doc_index)."""
    query_emb = model.encode(query, convert_to_tensor=True, normalize_embeddings=True)
    scores = util.cos_sim(query_emb, doc_embeddings)[0]  # shape: (N,)
    top_idx = torch_topk_indices(scores, k=top_k)

    results = []
    for i in top_idx:
        results.append((float(scores[i].item()), int(i)))
    return results


## 4) Padronizando resultados no “formato Elasticsearch”

No seu código real, `semantic_results` e `lexical_results` vêm do Elasticsearch com uma estrutura como:

```python
{
  "hits": {
    "hits": [
      {"_id": "...", "_score": 12.3, "_source": {...}},
      ...
    ]
  }
}
```

Para reaproveitar **a mesma lógica de RRF/WLF** com mínima mudança, vamos gerar resultados **nesse mesmo formato**, só que localmente.



In [13]:
def make_hit(doc_index: int, score: float):
    row = docs.iloc[doc_index]
    return {
        "_id": row["chunk_id"],   # no ES seria o _id do documento/chunk
        "_score": float(score),
        "_source": {
            "doc_id": int(row["doc_id"]),
            "chunk_id": row["chunk_id"],
            "title": row["title"],
            "content": row["content"],
            "source": "corpus_local",
            "file_path": None,
            "file_type": "text/plain",
            "created_at": None,
        }
    }

def wrap_hits(hits):
    return {"hits": {"hits": hits, "total": {"value": len(hits)}}}

def lexical_search_eslike(query: str, top_k: int = 10):
    scored = bm25.retrieve(query, top_k=top_k)
    hits = [make_hit(doc_index=i, score=s) for s, i in scored]
    return wrap_hits(hits)

def semantic_search_eslike(query: str, top_k: int = 10):
    scored = semantic_retrieve(query, top_k=top_k)
    hits = [make_hit(doc_index=i, score=s) for s, i in scored]
    return wrap_hits(hits)


## 5A) Implementando RRF (mesma ideia do código da aplicação)



In [14]:
def manual_rrf(semantic_results, lexical_results, k=60, top_k: int = 10):
    """Implementa Reciprocal Rank Fusion (RRF).

    RRF_score(d) = Σ 1/(k + rank_i(d))
    onde rank_i(d) é a posição (1-indexed) do documento d no ranking i.
    """
    from collections import defaultdict

    rrf_scores = defaultdict(float)
    doc_data = {}

    # Resultados semânticos
    for rank, hit in enumerate(semantic_results.get("hits", {}).get("hits", []), start=1):
        doc_id = hit["_id"]
        rrf_scores[doc_id] += 1.0 / (k + rank)
        if doc_id not in doc_data:
            doc_data[doc_id] = hit

    # Resultados léxicos
    for rank, hit in enumerate(lexical_results.get("hits", {}).get("hits", []), start=1):
        doc_id = hit["_id"]
        rrf_scores[doc_id] += 1.0 / (k + rank)
        if doc_id not in doc_data:
            doc_data[doc_id] = hit

    # Ordena por score final (maior primeiro)
    sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

    combined_hits = []
    for doc_id, score in sorted_docs[:top_k]:
        hit = doc_data[doc_id].copy()
        hit["_score"] = float(score)  # substitui o score pelo score RRF
        combined_hits.append(hit)

    return wrap_hits(combined_hits)


## 5B) Implementando WLF (mesma ideia do código da aplicação)



In [15]:
def manual_wlf(semantic_results, lexical_results, alpha=0.5, top_k: int = 10):
    """Implementa Weighted Linear Fusion (WLF).

    score_final(d) = alpha * norm_score_sem(d) + (1-alpha) * norm_score_lex(d)
    - norm_score_* aqui é score / max_score_da_lista (normalização simples).
    """
    from collections import defaultdict

    combined_scores = defaultdict(float)
    doc_data = {}

    # Normaliza scores semânticos
    semantic_hits = semantic_results.get("hits", {}).get("hits", [])
    if semantic_hits:
        max_sem = max(h["_score"] for h in semantic_hits) or 1.0
        for hit in semantic_hits:
            doc_id = hit["_id"]
            norm = hit["_score"] / max_sem
            combined_scores[doc_id] += alpha * norm
            if doc_id not in doc_data:
                doc_data[doc_id] = hit

    # Normaliza scores léxicos
    lexical_hits = lexical_results.get("hits", {}).get("hits", [])
    if lexical_hits:
        max_lex = max(h["_score"] for h in lexical_hits) or 1.0
        for hit in lexical_hits:
            doc_id = hit["_id"]
            norm = hit["_score"] / max_lex
            combined_scores[doc_id] += (1 - alpha) * norm
            if doc_id not in doc_data:
                doc_data[doc_id] = hit

    # Ordena por score combinado
    sorted_docs = sorted(combined_scores.items(), key=lambda x: x[1], reverse=True)

    combined_hits = []
    for doc_id, score in sorted_docs[:top_k]:
        hit = doc_data[doc_id].copy()
        hit["_score"] = float(score)
        combined_hits.append(hit)

    return wrap_hits(combined_hits)


## 5) Pipeline híbrido (sem ES) com RRF ou WLF

A função abaixo imita a sua assinatura mental:

- `method = "lexical" | "semantic" | "hybrid"`
- `rerank_method = "RRF" | "WLF"`
- `k` (RRF) e `alpha` (WLF) como hiperparâmetros



In [16]:
def pretty_table(es_response, label: str = None):
    hits = es_response.get("hits", {}).get("hits", [])
    rows = []
    for rank, h in enumerate(hits, start=1):
        content = h["_source"]["content"]
        rows.append({
            "rank": rank,
            "id": h["_id"],
            "score": float(h["_score"]),
            "title": h["_source"]["title"],
            "snippet": (content[:80] + ("..." if len(content) > 80 else "")),
        })
    df = pd.DataFrame(rows)
    if label:
        print(label)
    return df

def hybrid_search(query: str, method: str = "hybrid", rerank_method: str = "RRF",
                 k_rrf: int = 60, alpha_wlf: float = 0.5,
                 candidates_per_side: int = 10, top_k: int = 10):
    """Busca léxica, semântica ou híbrida + rerank (RRF/WLF)."""
    method = method.lower()
    rerank_method = rerank_method.upper()

    lexical_results = None
    semantic_results = None

    if method in ["lexical", "hybrid"]:
        lexical_results = lexical_search_eslike(query, top_k=candidates_per_side)

    if method in ["semantic", "hybrid"]:
        semantic_results = semantic_search_eslike(query, top_k=candidates_per_side)

    if method == "hybrid":
        if rerank_method == "RRF":
            results = manual_rrf(semantic_results, lexical_results, k=k_rrf, top_k=top_k)
        else:
            results = manual_wlf(semantic_results, lexical_results, alpha=alpha_wlf, top_k=top_k)
    elif method == "semantic":
        results = semantic_results
    else:
        results = lexical_results

    return results

# Teste rápido
query_demo = "como subir um serviço com docker compose?"
res_demo = hybrid_search(query_demo, method="hybrid", rerank_method="RRF")
display(pretty_table(res_demo, label=f"Consulta: {query_demo} (Híbrido + RRF)"))


Consulta: como subir um serviço com docker compose? (Híbrido + RRF)


,rank,id,score,title,snippet
0,1,17_0,0.032787,doc_17,Para subir o serviço: rode docker compose up -...
1,2,15_0,0.032258,doc_15,Manual do produto XJ-900: como calibrar e atua...
2,3,20_0,0.031258,doc_20,Comprei um novo teclado mecânico
3,4,18_0,0.015873,doc_18,"O contrato referência é o CT-2026-041, assinad..."
4,5,16_0,0.015625,doc_16,Erro E42 no sensor: verifique a conexão e rein...
5,6,5_0,0.015625,doc_5,Um cão estava correndo no parque
6,7,1_0,0.015385,doc_1,Um gato dorme em cima do sofá
7,8,19_0,0.015152,doc_19,A previsão do tempo indica chuva amanhã
8,9,6_0,0.015152,doc_6,O animal correu em um parque público
9,10,21_0,0.014925,doc_21,A fotossíntese ocorre nas folhas das plantas


## 6) Experimentos: comparando léxico, semântico, híbrido (RRF vs WLF)

A ideia é rodar a mesma consulta e observar:

- o que a busca léxica prioriza (termos exatos)
- o que a busca semântica prioriza (sentido)
- como a fusão resolve a disputa



In [17]:
queries = [
    "cão correndo no parque",
    "banco aumentou juros",              # instituição financeira
    "banco da praça",                    # assento
    "XJ-900 firmware",                   # termo/código exato
    "erro E42 sensor",                   # termo/código exato
    "gato dormindo no sofá em inglês",
]

def run_one_query(q: str, top_k: int = 5, candidates: int = 10, k_rrf: int = 60, alpha: float = 0.5):
    lex = hybrid_search(q, method="lexical", candidates_per_side=candidates, top_k=top_k)
    sem = hybrid_search(q, method="semantic", candidates_per_side=candidates, top_k=top_k)
    rrf = hybrid_search(q, method="hybrid", rerank_method="RRF", k_rrf=k_rrf, candidates_per_side=candidates, top_k=top_k)
    wlf = hybrid_search(q, method="hybrid", rerank_method="WLF", alpha_wlf=alpha, candidates_per_side=candidates, top_k=top_k)

    out = {
        "lexical (BM25)": pretty_table(lex),
        "semantic (embeddings)": pretty_table(sem),
        "hybrid + RRF": pretty_table(rrf),
        "hybrid + WLF": pretty_table(wlf),
    }
    return out

for q in queries:
    print("="*90)
    print("QUERY:", q)
    results = run_one_query(q, top_k=5, candidates=10, k_rrf=60, alpha=0.5)

    # Mostra lado a lado (colunas) usando concat com MultiIndex
    df = pd.concat(results, axis=1)
    display(df)


QUERY: cão correndo no parque


lexical (BM25)                          \
            rank    id     score   title   
0            1.0   5_0  9.427383   doc_5   
1            2.0   4_0  2.219395   doc_4   
2            3.0   6_0  1.935980   doc_6   
3            4.0   0_0  1.403091   doc_0   
4            5.0  13_0  1.313504  doc_13   
5            6.0   2_0  1.234671   doc_2   
6            7.0  16_0  1.046285  doc_16   
7            8.0  17_0  0.949683  doc_17   
8            NaN   NaN       NaN     NaN   
9            NaN   NaN       NaN     NaN   

                                                     semantic (embeddings)  \
                                             snippet                  rank   
0                   Um cão estava correndo no parque                     1   
1                      O cachorro correu pelo parque                     2   
2               O animal correu em um parque público                     3   
3                       O gato está dormindo no sofá                     4   
4            Sentei no banco da praça para descansar                     5   
5                Há um gato dormindo no sofá da sala                     6   
6  Erro E42 no sensor: verifique a conexão e rein...                     7   
7  Para subir o serviço: rode docker compose up -...                     8   
8                                                NaN                     9   
9                                                NaN                    10   

                                                                         \
     id     score   title                                       snippet   
0   5_0  0.986979   doc_5              Um cão estava correndo no parque   
1   7_0  0.978902   doc_7                The dog is running in the park   
2   4_0  0.965217   doc_4                 O cachorro correu pelo parque   
3   6_0  0.830045   doc_6          O animal correu em um parque público   
4  13_0  0.135357  doc_13       Sentei no banco da praça para descansar   
5  21_0  0.117739  doc_21  A fotossíntese ocorre nas folhas das plantas   
6  14_0  0.107606  doc_14      O banco central aumentou a taxa de juros   
7  19_0  0.106505  doc_19       A previsão do tempo indica chuva amanhã   
8  12_0  0.098106  doc_12                 O banco fechou mais cedo hoje   
9   8_0  0.077472   doc_8                Eu gosto de estudar matemática   

  hybrid + RRF                          \
          rank    id     score   title   
0          1.0   5_0  0.032787   doc_5   
1          2.0   4_0  0.032002   doc_4   
2          3.0   6_0  0.031498   doc_6   
3          4.0  13_0  0.030769  doc_13   
4          5.0   7_0  0.016129   doc_7   
5          NaN   NaN       NaN     NaN   
6          NaN   NaN       NaN     NaN   
7          NaN   NaN       NaN     NaN   
8          NaN   NaN       NaN     NaN   
9          NaN   NaN       NaN     NaN   

                                           hybrid + WLF                  \
                                   snippet         rank    id     score   
0         Um cão estava correndo no parque          1.0   5_0  1.000000   
1            O cachorro correu pelo parque          2.0   4_0  0.606686   
2     O animal correu em um parque público          3.0   6_0  0.523177   
3  Sentei no banco da praça para descansar          4.0   7_0  0.495909   
4           The dog is running in the park          5.0  13_0  0.138236   
5                                      NaN          NaN   NaN       NaN   
6                                      NaN          NaN   NaN       NaN   
7                                      NaN          NaN   NaN       NaN   
8                                      NaN          NaN   NaN       NaN   
9                                      NaN          NaN   NaN       NaN   

                                                    
    title                                  snippet  
0   doc_5         Um cão estava correndo no parque  
1   doc_4            O cachorro correu pelo parque  
2   doc_6     O a

QUERY: banco aumentou juros


lexical (BM25)                          \
            rank    id     score   title   
0            1.0  14_0  7.061098  doc_14   
1            2.0  12_0  2.068022  doc_12   
2            3.0  13_0  1.935980  doc_13   
3            NaN   NaN       NaN     NaN   
4            NaN   NaN       NaN     NaN   
5            NaN   NaN       NaN     NaN   
6            NaN   NaN       NaN     NaN   
7            NaN   NaN       NaN     NaN   
8            NaN   NaN       NaN     NaN   
9            NaN   NaN       NaN     NaN   

                                            semantic (embeddings)        \
                                    snippet                  rank    id   
0  O banco central aumentou a taxa de juros                     1  14_0   
1             O banco fechou mais cedo hoje                     2  12_0   
2   Sentei no banco da praça para descansar                     3  22_0   
3                                       NaN                     4  10_0   
4                                       NaN                     5   4_0   
5                                       NaN                     6   6_0   
6                                       NaN                     7   8_0   
7                                       NaN                     8   7_0   
8                                       NaN                     9   5_0   
9                                       NaN                    10   9_0   

                                                              hybrid + RRF  \
      score   title                                   snippet         rank   
0  0.838207  doc_14  O banco central aumentou a taxa de juros          1.0   
1  0.374158  doc_12             O banco fechou mais cedo hoje          2.0   
2  0.235625  doc_22      The stock market closed higher today          3.0   
3  0.123299  doc_10       Matemática é uma disciplina difícil          4.0   
4  0.121230   doc_4             O cachorro correu pelo parque          5.0   
5  0.117854   doc_6      O animal correu em um parque público          NaN   
6  0.108830   doc_8            Eu gosto de estudar matemática          NaN   
7  0.105788   doc_7            The dog is running in the park          NaN   
8  0.100912   doc_5          Um cão estava correndo no parque          NaN   
9  0.098792   doc_9          Aprender matemática é importante          NaN   

                                                                     \
     id     score   title                                   snippet   
0  14_0  0.032787  doc_14  O banco central aumentou a taxa de juros   
1  12_0  0.032258  doc_12             O banco fechou mais cedo hoje   
2  22_0  0.015873  doc_22      The stock market closed higher today   
3  13_0  0.015873  doc_13   Sentei no banco da praça para descansar   
4  10_0  0.015625  doc_10       Matemática é uma disciplina difícil   
5   NaN       NaN     NaN                                       NaN   
6   NaN       NaN     NaN                                       NaN   
7   NaN       NaN     NaN                                       NaN   
8   NaN       NaN     NaN                                       NaN   
9   NaN       NaN     NaN                                       NaN   

  hybrid + WLF                          \
          rank    id     score   title   
0          1.0  14_0  1.000000  doc_14   
1          2.0  12_0  0.369628  doc_12   
2          3.0  22_0  0.140553  doc_22   
3          4.0  13_0  0.137088  doc_13   
4          5.0  10_0  0.073549  doc_10   
5          NaN   NaN       NaN     NaN   
6          NaN   NaN       NaN     NaN   
7          NaN   NaN       NaN     NaN   
8          NaN   NaN       NaN     NaN   
9          NaN   NaN       NaN     NaN   

                                             
                                    snippet  
0  O banco central aumentou a taxa de juros  
1             O banco fechou mais cedo hoje  
2      The stock market closed higher today  
3   Sentei no banco da praça para descansar  
4

QUERY: banco da praça


...

lexical (BM25)                          \
            rank    id     score   title   
0            1.0  15_0  6.662388  doc_15   
1            NaN   NaN       NaN     NaN   
2            NaN   NaN       NaN     NaN   
3            NaN   NaN       NaN     NaN   
4            NaN   NaN       NaN     NaN   
5            NaN   NaN       NaN     NaN   
6            NaN   NaN       NaN     NaN   
7            NaN   NaN       NaN     NaN   
8            NaN   NaN       NaN     NaN   
9            NaN   NaN       NaN     NaN   

                                                     semantic (embeddings)  \
                                             snippet                  rank   
0  Manual do produto XJ-900: como calibrar e atua...                     1   
1                                                NaN                     2   
2                                                NaN                     3   
3                                                NaN                     4   
4                                                NaN                     5   
5                                                NaN                     6   
6                                                NaN                     7   
7                                                NaN                     8   
8                                                NaN                     9   
9                                                NaN                    10   

                                                                              \
     id     score   title                                            snippet   
0  15_0  0.893836  doc_15  Manual do produto XJ-900: como calibrar e atua...   
1  16_0  0.428350  doc_16  Erro E42 no sensor: verifique a conexão e rein...   
2  20_0  0.336789  doc_20                   Comprei um novo teclado mecânico   
3  18_0  0.219418  doc_18  O contrato referência é o CT-2026-041, assinad...   
4  17_0  0.202165  doc_17  Para subir o serviço: rode docker compose up -...   
5  12_0  0.102660  doc_12                      O banco fechou mais cedo hoje   
6  22_0  0.090517  doc_22               The stock market closed higher today   
7   7_0  0.043345   doc_7                     The dog is running in the park   
8   5_0  0.035841   doc_5                   Um cão estava correndo no parque   
9   8_0  0.035142   doc_8                     Eu gosto de estudar matemática   

  hybrid + RRF                          \
          rank    id     score   title   
0          1.0  15_0  0.032787  doc_15   
1          2.0  16_0  0.016129  doc_16   
2          3.0  20_0  0.015873  doc_20   
3          4.0  18_0  0.015625  doc_18   
4          5.0  17_0  0.015385  doc_17   
5          NaN   NaN       NaN     NaN   
6          NaN   NaN       NaN     NaN   
7          NaN   NaN       NaN     NaN   
8          NaN   NaN       NaN     NaN   
9          NaN   NaN       NaN     NaN   

                                                     hybrid + WLF        \
                                             snippet         rank    id   
0  Manual do produto XJ-900: como calibrar e atua...          1.0  15_0   
1  Erro E42 no sensor: verifique a conexão e rein...          2.0  16_0   
2                   Comprei um novo teclado mecânico          3.0  20_0   
3  O contrato referência é o CT-2026-041, assinad...          4.0  18_0   
4  Para subir o serviço: rode docker compose up -...          5.0  17_0   
5                                                NaN          NaN   NaN   
6                                                NaN          NaN   NaN   
7                                                NaN          NaN   NaN   
8                                                NaN          NaN   NaN   
9                                                NaN          NaN   NaN   

                                                                        
      score   title                                            snippet  
0  1.000000  doc_15  Ma

QUERY: erro E42 sensor


lexical (BM25)                          \
            rank    id     score   title   
0            1.0  16_0  6.662388  doc_16   
1            NaN   NaN       NaN     NaN   
2            NaN   NaN       NaN     NaN   
3            NaN   NaN       NaN     NaN   
4            NaN   NaN       NaN     NaN   
5            NaN   NaN       NaN     NaN   
6            NaN   NaN       NaN     NaN   
7            NaN   NaN       NaN     NaN   
8            NaN   NaN       NaN     NaN   
9            NaN   NaN       NaN     NaN   

                                                     semantic (embeddings)  \
                                             snippet                  rank   
0  Erro E42 no sensor: verifique a conexão e rein...                     1   
1                                                NaN                     2   
2                                                NaN                     3   
3                                                NaN                     4   
4                                                NaN                     5   
5                                                NaN                     6   
6                                                NaN                     7   
7                                                NaN                     8   
8                                                NaN                     9   
9                                                NaN                    10   

                                                                              \
     id     score   title                                            snippet   
0  16_0  0.857701  doc_16  Erro E42 no sensor: verifique a conexão e rein...   
1  15_0  0.397445  doc_15  Manual do produto XJ-900: como calibrar e atua...   
2  18_0  0.242199  doc_18  O contrato referência é o CT-2026-041, assinad...   
3  20_0  0.192941  doc_20                   Comprei um novo teclado mecânico   
4  17_0  0.145988  doc_17  Para subir o serviço: rode docker compose up -...   
5  19_0  0.118160  doc_19            A previsão do tempo indica chuva amanhã   
6   9_0  0.048529   doc_9                   Aprender matemática é importante   
7  10_0  0.042881  doc_10                Matemática é uma disciplina difícil   
8  22_0  0.024524  doc_22               The stock market closed higher today   
9  13_0  0.022983  doc_13            Sentei no banco da praça para descansar   

  hybrid + RRF                          \
          rank    id     score   title   
0          1.0  16_0  0.032787  doc_16   
1          2.0  15_0  0.016129  doc_15   
2          3.0  18_0  0.015873  doc_18   
3          4.0  20_0  0.015625  doc_20   
4          5.0  17_0  0.015385  doc_17   
5          NaN   NaN       NaN     NaN   
6          NaN   NaN       NaN     NaN   
7          NaN   NaN       NaN     NaN   
8          NaN   NaN       NaN     NaN   
9          NaN   NaN       NaN     NaN   

                                                     hybrid + WLF        \
                                             snippet         rank    id   
0  Erro E42 no sensor: verifique a conexão e rein...          1.0  16_0   
1  Manual do produto XJ-900: como calibrar e atua...          2.0  15_0   
2  O contrato referência é o CT-2026-041, assinad...          3.0  18_0   
3                   Comprei um novo teclado mecânico          4.0  20_0   
4  Para subir o serviço: rode docker compose up -...          5.0  17_0   
5                                                NaN          NaN   NaN   
6                                                NaN          NaN   NaN   
7                                                NaN          NaN   NaN   
8                                                NaN          NaN   NaN   
9                                                NaN          NaN   NaN   

                                                                        
      score   title                                            snippet  
0  1.000000  doc_16  Er

QUERY: gato dormindo no sofá em inglês


lexical (BM25)                          \
            rank    id     score   title   
0              1   0_0  7.698629   doc_0   
1              2   2_0  6.774526   doc_2   
2              3   1_0  5.555231   doc_1   
3              4   6_0  1.935980   doc_6   
4              5   3_0  1.683271   doc_3   
5              6  18_0  1.542125  doc_18   
6              7   5_0  1.403091   doc_5   
7              8  13_0  1.313504  doc_13   
8              9  16_0  1.046285  doc_16   
9             10  17_0  0.949683  doc_17   

                                                     semantic (embeddings)  \
                                             snippet                  rank   
0                       O gato está dormindo no sofá                     1   
1                Há um gato dormindo no sofá da sala                     2   
2                      Um gato dorme em cima do sofá                     3   
3               O animal correu em um parque público                     4   
4                    The cat is sleeping on the sofa                     5   
5  O contrato referência é o CT-2026-041, assinad...                     6   
6                   Um cão estava correndo no parque                     7   
7            Sentei no banco da praça para descansar                     8   
8  Erro E42 no sensor: verifique a conexão e rein...                     9   
9  Para subir o serviço: rode docker compose up -...                    10   

                                                                              \
     id     score   title                                            snippet   
0   1_0  0.922315   doc_1                      Um gato dorme em cima do sofá   
1   0_0  0.909589   doc_0                       O gato está dormindo no sofá   
2   2_0  0.903777   doc_2                Há um gato dormindo no sofá da sala   
3   3_0  0.899378   doc_3                    The cat is sleeping on the sofa   
4  13_0  0.230836  doc_13            Sentei no banco da praça para descansar   
5  20_0  0.126155  doc_20                   Comprei um novo teclado mecânico   
6  14_0  0.079636  doc_14           O banco central aumentou a taxa de juros   
7   8_0  0.070929   doc_8                     Eu gosto de estudar matemática   
8  18_0  0.056765  doc_18  O contrato referência é o CT-2026-041, assinad...   
9  22_0  0.013089  doc_22               The stock market closed higher today   

  hybrid + RRF                          \
          rank    id     score   title   
0          1.0   0_0  0.032522   doc_0   
1          2.0   1_0  0.032266   doc_1   
2          3.0   2_0  0.032002   doc_2   
3          4.0   3_0  0.031010   doc_3   
4          5.0  13_0  0.030090  doc_13   
5          NaN   NaN       NaN     NaN   
6          NaN   NaN       NaN     NaN   
7          NaN   NaN       NaN     NaN   
8          NaN   NaN       NaN     NaN   
9          NaN   NaN       NaN     NaN   

                                           hybrid + WLF                  \
                                   snippet         rank    id     score   
0             O gato está dormindo no sofá          1.0   0_0  0.993101   
1            Um gato dorme em cima do sofá          2.0   2_0  0.929933   
2      Há um gato dormindo no sofá da sala          3.0   1_0  0.860794   
3          The cat is sleeping on the sofa          4.0   3_0  0.596888   
4  Sentei no banco da praça para descansar          5.0  13_0  0.210447   
5                                      NaN          NaN   NaN       NaN   
6                                      NaN          NaN   NaN       NaN   
7                                      NaN          NaN   NaN       NaN   
8                                      NaN          NaN   NaN       NaN   
9                                      NaN          NaN   NaN       NaN   

                                                    
    title                                  snippet  
0   doc_0             O gato está dormindo no sofá  
1   doc_2 

## 7) Ajuste de hiperparâmetros (k e alpha)

Experimente:

- **RRF**:
  - `k` menor → posições do topo valem mais (ranking “mais agressivo”)
  - `k` maior → diferenças de rank pesam menos (mais “suave”)

- **WLF**:
  - `alpha` maior → você confia mais na semântica
  - `alpha` menor → você confia mais na léxica

Abaixo você pode mexer nesses valores e ver o ranking mudar.



In [18]:
# Brinque com esses valores:
QUERY = "banco juros"
K_RRF = 60
ALPHA = 0.6
CANDIDATES = 12
TOPK = 8

lex = hybrid_search(QUERY, method="lexical", candidates_per_side=CANDIDATES, top_k=TOPK)
sem = hybrid_search(QUERY, method="semantic", candidates_per_side=CANDIDATES, top_k=TOPK)
rrf = hybrid_search(QUERY, method="hybrid", rerank_method="RRF", k_rrf=K_RRF, candidates_per_side=CANDIDATES, top_k=TOPK)
wlf = hybrid_search(QUERY, method="hybrid", rerank_method="WLF", alpha_wlf=ALPHA, candidates_per_side=CANDIDATES, top_k=TOPK)

display(pd.concat({
    "lexical": pretty_table(lex),
    "semantic": pretty_table(sem),
    f"hybrid_RRF(k={K_RRF})": pretty_table(rrf),
    f"hybrid_WLF(alpha={ALPHA})": pretty_table(wlf),
}, axis=1))


lexical                                                                    \
      rank    id     score   title                                   snippet   
0      1.0  14_0  4.440443  doc_14  O banco central aumentou a taxa de juros   
1      2.0  12_0  2.068022  doc_12             O banco fechou mais cedo hoje   
2      3.0  13_0  1.935980  doc_13   Sentei no banco da praça para descansar   
3      NaN   NaN       NaN     NaN                                       NaN   
4      NaN   NaN       NaN     NaN                                       NaN   
5      NaN   NaN       NaN     NaN                                       NaN   
6      NaN   NaN       NaN     NaN                                       NaN   
7      NaN   NaN       NaN     NaN                                       NaN   
8      NaN   NaN       NaN     NaN                                       NaN   
9      NaN   NaN       NaN     NaN                                       NaN   
10     NaN   NaN       NaN     NaN                                       NaN   
11     NaN   NaN       NaN     NaN                                       NaN   

   semantic                          \
       rank    id     score   title   
0         1  14_0  0.443372  doc_14   
1         2  12_0  0.420730  doc_12   
2         3  18_0  0.234877  doc_18   
3         4  10_0  0.198462  doc_10   
4         5   9_0  0.179314   doc_9   
5         6  20_0  0.156260  doc_20   
6         7  19_0  0.142514  doc_19   
7         8   8_0  0.132337   doc_8   
8         9  13_0  0.122548  doc_13   
9        10  22_0  0.118111  doc_22   
10       11  17_0  0.105019  doc_17   
11       12  11_0  0.099644  doc_11   

                                                      hybrid_RRF(k=60)        \
                                              snippet             rank    id   
0            O banco central aumentou a taxa de juros              1.0  14_0   
1                       O banco fechou mais cedo hoje              2.0  12_0   
2   O contrato referência é o CT-2026-041, assinad...              3.0  13_0   
3                 Matemática é uma disciplina difícil              4.0  18_0   
4                    Aprender matemática é importante              5.0  10_0   
5                    Comprei um novo teclado mecânico              6.0   9_0   
6             A previsão do tempo indica chuva amanhã              7.0  20_0   
7                      Eu gosto de estudar matemática              8.0  19_0   
8             Sentei no banco da praça para descansar              NaN   NaN   
9                The stock market closed higher today              NaN   NaN   
10  Para subir o serviço: rode docker compose up -...              NaN   NaN   
11                 Ensinar matemática exige paciência              NaN   NaN   

                                                                         \
       score   title                                            snippet   
0   0.032787  doc_14           O banco central aumentou a taxa de juros   
1   0.032258  doc_12                      O banco fechou mais cedo hoje   
2   0.030366  doc_13            Sentei no banco da praça para descansar   
3   0.015873  doc_18  O contrato referência é o CT-2026-041, assinad...   
4   0.015625  doc_10                Matemática é uma disciplina difícil   
5   0.015385   doc_9                   Aprender matemática é importante   
6   0.015152  doc_20                   Comprei um novo teclado mecânico   
7   0.014925  doc_19            A previsão do tempo indica chuva amanhã   
8        NaN     NaN                                                NaN   
9        NaN     NaN                                                NaN   
10       NaN     NaN                                                NaN   
11       NaN     NaN                                                NaN   

   hybrid_WLF(alpha=0.6)                          \
                    rank    id     score   title   
0                    1.0  14_0  1.000000  doc_14   
1      

## 8) Exercícios sugeridos

1) **Aumente o corpus**  
   - Adicione 10 frases novas (incluindo sinônimos e termos exatos)

2) **Teste consultas “difíceis”**  
   - Parafraseie sem usar palavras do documento (semântica deve ajudar)
   - Use códigos exatos (léxica deve dominar)

3) **Explore hiperparâmetros**  
   - RRF: compare `k=10`, `k=60`, `k=200`
   - WLF: compare `alpha=0.2`, `0.5`, `0.8`

4) **Pergunta reflexiva**  
   - Quando a RRF parece melhor que a WLF?  
   - Quando a WLF parece melhor que a RRF?  
   *Dica*: pense em “escala” e “calibração” de scores.

Se quiser, no próximo notebook dá para adicionar uma métrica simples (ex.: *Precision@k*) para avaliar automaticamente.

